# Stage 06-B: Decision-Level Fusion — Independent Branch Stacking

**Motivation:** All previous notebooks fuse text and image at the *feature* level (concatenate or
gate intermediate representations). Decision-level fusion trains two completely independent classifiers
and combines their *output logits*. This tests whether the two modalities carry complementary
decision signals that are best combined after each branch specialises.

## Architecture

```
PhoBERT CLS [B, 768]              COOLANT image [B, 64] (mean-pooled)
         │                                  │
  TextBranch                        ImageBranch
  Linear[768→256]+LN+ReLU           Linear[64→64]+LN+ReLU
  Dropout(0.3)                       Dropout(0.3)
  Linear[256→3]                      Linear[64→3]
  text_logits [B, 3]                 img_logits [B, 3]
         │                                  │
         └──────── cat → [B, 6] ────────────┘
                          │
               MetaLearner: Linear[6→3]
               (learns to weight and recalibrate both branches)
                          │
               final_logits [B, 3]  (Real=0, Fake=1, NEI=2)
```

## Training with Auxiliary Losses

Each branch is regularised with its own ground-truth loss:
```
L = L_main(final_logits, y)  +  aux_weight * L_aux(text_logits, y)
                             +  aux_weight * L_aux(img_logits, y)
```
This forces each branch to be independently discriminative, not free-ride on the meta-learner.

## Key Difference vs Feature Fusion (05b)

| | 05b Asym Gate | 06b Decision Fusion |
|---|---|---|
| Fusion level | Feature (hidden [256]) | Decision (logits [3]) |
| Modality coupling | gate mixes representations | branches fully independent |
| Interpretability | gate value shows text/image weight | logits per branch directly interpretable |
| Branch loss | shared single loss | auxiliary losses per branch |

**Inputs:**
- `training/stage05b_cache/stage05b_{split}.h5` — reuses 05b mean-pooled cache (no rebuild)

**Outputs:**
- `training/checkpoints_stage06b/{timestamp}_decision-fusion_phobert768-coolant64_3cls_bs{B}_lr{lr}/best_model.pth`
- `training/stage06b_results/{timestamp}_results.json`


In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True

PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()

print(f'Platform : {PLATFORM}  Project: {PROJECT_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}  exists={DATA_ROOT.exists()}')

## Step 1: Configuration


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'pipeline' else Path.cwd()
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env', override=False)
except ImportError:
    pass
DATA_ROOT = Path(os.environ['DATA_ROOT']) if os.environ.get('DATA_ROOT') else PROJECT_ROOT

CONFIG = {
    'paths': {
        'project_root':            PROJECT_ROOT,
        # Reuse Stage 05b mean-pooled cache — text[768] + image[64]
        'stage05b_cache_dir':      DATA_ROOT / 'training' / 'stage05b_cache',
        'stage39_checkpoint_root': DATA_ROOT / 'training' / 'checkpoints_vifactcheck',
        'stage39_manifest':        None,
        'checkpoint_root':         DATA_ROOT / 'training' / 'checkpoints_stage06b',
        'results_dir':             DATA_ROOT / 'training' / 'stage06b_results',
        'mlflow_dir':              DATA_ROOT / 'mlruns',
    },
    'model': {
        'arch_tag':    'decision-fusion_phobert768-coolant64_3cls',
        'text_dim':    768,   # PhoBERT CLS
        'image_dim':   64,    # COOLANT image_aligned_clip (mean-pooled)
        'text_hidden': 256,   # text branch hidden dim
        'img_hidden':  64,    # image branch hidden dim (small — image dim is small)
        'num_classes': 3,     # Real=0, Fake=1, NEI=2
        'dropout':     0.3,
    },
    'training': {
        'batch_size':          32,
        'max_epochs':          40,
        'patience':            8,
        'lr':                  3e-4,
        'weight_decay':        1e-4,
        'label_smoothing':     0.1,
        # Auxiliary loss weight per branch (0 = disable, recommended: 0.2–0.4)
        # Auxiliary losses keep each branch independently discriminative
        'aux_weight':          0.3,
        'grad_clip':           1.0,
        'onecycle_pct_start':  0.05,
        'seed':                42,
    },
    'mlflow': {'experiment_name': 'stage06b-decision-fusion'},
    'safety': {
        'smoke_test':    False,
        'smoke_batches': 3,
        'smoke_epochs':  2,
    },
}

CONFIG['paths']['checkpoint_root'].mkdir(parents=True, exist_ok=True)
CONFIG['paths']['results_dir'].mkdir(parents=True, exist_ok=True)

print(f'Checkpoint root : {CONFIG["paths"]["checkpoint_root"]}')
print(f'Stage05b cache  : {CONFIG["paths"]["stage05b_cache_dir"]}')
print(f'Arch tag        : {CONFIG["model"]["arch_tag"]}')
print(f'aux_weight      : {CONFIG["training"]["aux_weight"]}')

## Step 2: Dependencies, Device, Seed


In [ ]:
import importlib, sys, random, json, gc
from datetime import datetime

_required = [
    ('torch', 'torch'), ('h5py', 'h5py'), ('numpy', 'numpy'),
    ('pandas', 'pandas'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'),
    ('tqdm', 'tqdm'), ('sklearn', 'scikit-learn'), ('mlflow', 'mlflow'),
]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError(f'Missing packages: {_missing}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def select_device():
    from src.utils.device import get_device
    return get_device()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    print(f'Seed: {seed}')

device = select_device()
seed_everything(CONFIG['training']['seed'])
print(f'Device: {device}')

## Step 3: Validate Stage 05b Cache

Reuses `stage05b_cache` built by `05b_asymmetric_gated_fusion.ipynb` (Step 4).
The cache already has mean-pooled image features — no rebuild needed.


In [ ]:
cache_dir = CONFIG['paths']['stage05b_cache_dir']

for split in ('train', 'dev', 'test'):
    p = cache_dir / f'stage05b_{split}.h5'
    if not p.exists():
        raise FileNotFoundError(
            f'Stage 05b cache missing: {p}\n'
            'Run 05b_asymmetric_gated_fusion.ipynb first (Step 4 builds the cache).'
        )
    with h5py.File(p, 'r') as f:
        n     = f.attrs['n_articles']
        t_dim = f.attrs.get('text_dim', 768)
        i_dim = f.attrs.get('image_dim', 64)
        hist  = f.attrs.get('label_hist', '?')
        print(f'  [{split}] n={n}  text_dim={t_dim}  image_dim={i_dim}  label_hist={hist}')

print('\u2705 Stage 05b cache validated.')

## Step 4: Dataset and DataLoaders


In [ ]:
class ArticleDataset(Dataset):
    """Loads stage05b cache: text[768], image[64] (mean-pooled), label."""
    def __init__(self, h5_path):
        with h5py.File(h5_path, 'r') as f:
            self.text_features  = f['text_features'][:].astype(np.float32)   # [N, 768]
            self.image_features = f['image_features'][:].astype(np.float32)  # [N, 64]
            self.labels         = f['labels'][:].astype(np.int64)             # [N]
        print(f'  [{Path(h5_path).stem}] {len(self.labels)} articles')

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.text_features[idx]),
            torch.from_numpy(self.image_features[idx]),
            int(self.labels[idx]),
        )

datasets = {s: ArticleDataset(cache_dir / f'stage05b_{s}.h5') for s in ('train', 'dev', 'test')}
bs = CONFIG['training']['batch_size']
loaders = {
    'train': DataLoader(datasets['train'], batch_size=bs, shuffle=True,  num_workers=0),
    'dev':   DataLoader(datasets['dev'],   batch_size=bs, shuffle=False, num_workers=0),
    'test':  DataLoader(datasets['test'],  batch_size=bs, shuffle=False, num_workers=0),
}

if CONFIG['safety']['smoke_test']:
    from itertools import islice
    class _Smoke:
        def __init__(self, l, n): self._l, self._n = l, n
        def __iter__(self): return islice(iter(self._l), self._n)
        def __len__(self): return min(self._n, len(self._l))
    loaders = {k: _Smoke(v, CONFIG['safety']['smoke_batches']) for k, v in loaders.items()}

_t, _img, _lbl = next(iter(DataLoader(datasets['train'], batch_size=4)))
print(f'Batch: text={tuple(_t.shape)}  image={tuple(_img.shape)}  labels={_lbl.tolist()}')
print(f'Train label hist: {np.bincount(datasets["train"].labels, minlength=3).tolist()}')

## Step 5: DecisionFusionHead

Two fully independent branches produce 3-class logits. A small `MetaLearner` (Linear[6→3]) learns
how to weight and recalibrate both branches for the final prediction.

The branch logits are passed **without** softmax so the meta-learner receives raw logit magnitudes
(not normalised probabilities), which preserves confidence information.


In [ ]:
class DecisionFusionHead(nn.Module):
    """
    Decision-level fusion via independent branch stacking.

    Inputs:
        text_feat  [B, 768] — frozen PhoBERT CLS (Stage 3.9)
        image_feat [B, 64]  — frozen COOLANT image_aligned_clip, mean-pooled (Stage 2)
    Outputs:
        final_logits [B, 3]  — combined prediction
        text_logits  [B, 3]  — text-only branch (for auxiliary loss + analysis)
        img_logits   [B, 3]  — image-only branch (for auxiliary loss + analysis)
    """

    def __init__(self, text_dim=768, image_dim=64,
                 text_hidden=256, img_hidden=64,
                 num_classes=3, dropout=0.3):
        super().__init__()
        # Text branch: richer because PhoBERT is much higher-dim
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, text_hidden),
            nn.LayerNorm(text_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(text_hidden, num_classes),
        )
        # Image branch: shallower — 64-dim input already compact
        self.image_branch = nn.Sequential(
            nn.Linear(image_dim, img_hidden),
            nn.LayerNorm(img_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(img_hidden, num_classes),
        )
        # Meta-learner: combines raw logits from both branches
        # Receives [text_logits(3), img_logits(3)] = 6-dim input
        self.meta = nn.Linear(num_classes * 2, num_classes, bias=True)

        # Initialise meta weights: slight prior toward text (stronger modality)
        with torch.no_grad():
            nn.init.constant_(self.meta.weight[:, :num_classes], 0.7 / num_classes)
            nn.init.constant_(self.meta.weight[:, num_classes:], 0.3 / num_classes)
            nn.init.zeros_(self.meta.bias)

    def forward(self, text_feat, image_feat):
        t_logits = self.text_branch(text_feat)    # [B, 3]
        i_logits = self.image_branch(image_feat)  # [B, 3]
        combined = torch.cat([t_logits, i_logits], dim=-1)  # [B, 6]
        final    = self.meta(combined)            # [B, 3]
        return final, t_logits, i_logits


# ── Instantiate ────────────────────────────────────────────────────────────
head = DecisionFusionHead(
    text_dim=CONFIG['model']['text_dim'],
    image_dim=CONFIG['model']['image_dim'],
    text_hidden=CONFIG['model']['text_hidden'],
    img_hidden=CONFIG['model']['img_hidden'],
    num_classes=CONFIG['model']['num_classes'],
    dropout=CONFIG['model']['dropout'],
).to(device)

total_p     = sum(p.numel() for p in head.parameters())
trainable_p = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f'DecisionFusionHead — total: {total_p:,}  trainable: {trainable_p:,}')

# Param breakdown
for name, mod in head.named_children():
    p_cnt = sum(p.numel() for p in mod.parameters())
    print(f'  {name}: {p_cnt:,}')

head.eval()
with torch.no_grad():
    _final, _tl, _il = head(_t[:2].to(device), _img[:2].to(device))
print(f'\nSanity — final:{tuple(_final.shape)}  text_logits:{tuple(_tl.shape)}  img_logits:{tuple(_il.shape)}')
print(f'  text_logits sample: {_tl[0].cpu().numpy().round(3)}')
print(f'  img_logits  sample: {_il[0].cpu().numpy().round(3)}')
head.train()

## Step 6: Loss, Optimizer, Scheduler


In [ ]:
nc = CONFIG['model']['num_classes']
train_labels = datasets['train'].labels
counts = np.bincount(train_labels, minlength=nc).astype(np.float64)
class_weights = torch.tensor(len(train_labels) / (nc * counts), dtype=torch.float32).to(device)
print(f'Class counts : {counts.astype(int).tolist()}')
print(f'Class weights: {[round(w, 4) for w in class_weights.cpu().tolist()]}')

# Main loss for combined head
loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG['training']['label_smoothing'],
)
# Auxiliary loss for each branch (no label smoothing — we want raw branch signal)
aux_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    head.parameters(),
    lr=CONFIG['training']['lr'],
    weight_decay=CONFIG['training']['weight_decay'],
)

total_steps = CONFIG['training']['max_epochs'] * len(loaders['train'])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['training']['lr'],
    total_steps=total_steps,
    pct_start=CONFIG['training']['onecycle_pct_start'],
    anneal_strategy='cos',
)
print(f'Optimizer: AdamW  lr={CONFIG["training"]["lr"]}')
print(f'aux_weight={CONFIG["training"]["aux_weight"]}  total_steps={total_steps}')

## Step 7: Checkpoint Setup


In [ ]:
import mlflow

def _lr_to_str(lr):
    return f'{lr:.0e}'.replace('+0', '').replace('-0', '-').replace('e0', 'e')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
arch_tag  = CONFIG['model']['arch_tag']
bs        = CONFIG['training']['batch_size']
lr_str    = _lr_to_str(CONFIG['training']['lr'])
run_name  = f'{timestamp}_{arch_tag}_bs{bs}_lr{lr_str}'
if CONFIG['safety']['smoke_test']: run_name = f'SMOKE_{run_name}'

run_dir = CONFIG['paths']['checkpoint_root'] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = run_dir / 'best_model.pth'
print(f'Run dir: {run_dir}')

def _cfg_jsonable(cfg):
    if isinstance(cfg, dict):          return {k: _cfg_jsonable(v) for k, v in cfg.items()}
    if isinstance(cfg, Path):          return str(cfg)
    if isinstance(cfg, (list, tuple)): return [_cfg_jsonable(v) for v in cfg]
    return cfg

def save_checkpoint(path, model, epoch, metrics, is_best):
    torch.save({
        'model_state_dict': model.state_dict(),
        'config':       _cfg_jsonable(CONFIG),
        'epoch':        epoch,
        'metrics':      metrics,
        'arch_tag':     CONFIG['model']['arch_tag'],
        'text_dim':     CONFIG['model']['text_dim'],
        'image_dim':    CONFIG['model']['image_dim'],
        'text_hidden':  CONFIG['model']['text_hidden'],
        'img_hidden':   CONFIG['model']['img_hidden'],
        'num_classes':  CONFIG['model']['num_classes'],
        'is_best':      is_best,
        'run_name':     run_name,
    }, path)

mlflow_enabled = False
try:
    mlflow.set_tracking_uri(CONFIG['paths']['mlflow_dir'].as_uri())
    mlflow.set_experiment(CONFIG['mlflow']['experiment_name'])
    _run = mlflow.start_run(run_name=run_name)
    mlflow.log_params({
        'arch_tag': arch_tag, 'aux_weight': CONFIG['training']['aux_weight'],
        'text_hidden': CONFIG['model']['text_hidden'],
        'img_hidden': CONFIG['model']['img_hidden'],
        'batch_size': bs, 'lr': CONFIG['training']['lr'],
    })
    mlflow_enabled = True
    print(f'MLflow run: {_run.info.run_id}')
except Exception as e:
    print(f'MLflow disabled ({type(e).__name__})')

## Step 8: Training Loop with Auxiliary Branch Losses

Each batch computes three losses:
1. `L_main` — final combined prediction  
2. `L_text` — text branch alone (× `aux_weight`)  
3. `L_img`  — image branch alone (× `aux_weight`)  

This prevents the meta-learner from ignoring a weak branch — both must contribute.


In [ ]:
def compute_metrics(y_true, y_pred, prefix, nc):
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
    m = {f'{prefix}_accuracy': round(acc, 4), f'{prefix}_macro_f1': round(macro_f1, 4)}
    for i, cls in enumerate(['real', 'fake', 'nei']):
        m[f'{prefix}_f1_{cls}'] = round(float(per_f1[i]) if i < len(per_f1) else 0.0, 4)
    return m

def run_epoch(model, loader, loss_fn, aux_loss_fn, optimizer, scheduler, device, train, aux_weight):
    model.train() if train else model.eval()
    total_loss, total_main, total_aux, n_batches = 0.0, 0.0, 0.0, 0
    y_true_all, y_pred_all = [], []
    # For eval: track per-branch accuracy
    y_pred_text, y_pred_img = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc='  train' if train else '  eval ', leave=False):
            text_feat, image_feat, labels = batch
            text_feat  = text_feat.to(device)
            image_feat = image_feat.to(device)
            labels     = labels.to(device)

            final_logits, t_logits, i_logits = model(text_feat, image_feat)

            l_main = loss_fn(final_logits, labels)
            l_text = aux_loss_fn(t_logits, labels)
            l_img  = aux_loss_fn(i_logits, labels)
            loss   = l_main + aux_weight * l_text + aux_weight * l_img

            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss: {loss.item()}')

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['training']['grad_clip'])
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            total_loss += loss.item()
            total_main += l_main.item()
            total_aux  += (l_text.item() + l_img.item()) / 2
            n_batches  += 1

            y_pred_all.extend(final_logits.argmax(-1).cpu().tolist())
            y_true_all.extend(labels.cpu().tolist())
            y_pred_text.extend(t_logits.argmax(-1).cpu().tolist())
            y_pred_img.extend(i_logits.argmax(-1).cpu().tolist())

    B = max(1, n_batches)
    branch_acc_text = float(accuracy_score(y_true_all, y_pred_text))
    branch_acc_img  = float(accuracy_score(y_true_all, y_pred_img))
    return (
        total_loss / B, total_main / B, total_aux / B,
        np.array(y_true_all), np.array(y_pred_all),
        branch_acc_text, branch_acc_img,
    )


best_val_f1 = -1.0; best_epoch = -1; patience_ctr = 0; history = []
aux_weight  = CONFIG['training']['aux_weight']
max_epochs  = CONFIG['safety']['smoke_epochs'] if CONFIG['safety']['smoke_test'] else CONFIG['training']['max_epochs']

print(f'Training: max_epochs={max_epochs}  patience={CONFIG["training"]["patience"]}  aux_weight={aux_weight}')

for epoch in range(1, max_epochs + 1):
    tr_loss, tr_main, tr_aux, yt, yp, tr_t_acc, tr_i_acc = run_epoch(
        head, loaders['train'], loss_fn, aux_loss_fn, optimizer, scheduler, device, True, aux_weight)
    vl_loss, vl_main, vl_aux, yv, yp_v, vl_t_acc, vl_i_acc = run_epoch(
        head, loaders['dev'],   loss_fn, aux_loss_fn, None,      None,      device, False, aux_weight)

    tr_m = compute_metrics(yt, yp,   'train', nc)
    vl_m = compute_metrics(yv, yp_v, 'val',   nc)

    row = {
        'epoch': epoch,
        'train_loss': round(tr_loss, 4), 'val_loss': round(vl_loss, 4),
        'train_main_loss': round(tr_main, 4), 'val_main_loss': round(vl_main, 4),
        'val_text_branch_acc': round(vl_t_acc, 4),
        'val_img_branch_acc':  round(vl_i_acc, 4),
        **tr_m, **vl_m,
    }
    history.append(row)

    if mlflow_enabled:
        try: mlflow.log_metrics({'train_loss': tr_loss, 'val_loss': vl_loss,
                                  'val_text_branch_acc': vl_t_acc,
                                  'val_img_branch_acc': vl_i_acc, **tr_m, **vl_m}, step=epoch)
        except Exception: pass

    val_f1 = vl_m['val_macro_f1']
    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1; best_epoch = epoch; patience_ctr = 0
        save_checkpoint(best_ckpt_path, head, epoch, vl_m, is_best=True)
        marker = ' \u2190 best'
    else:
        patience_ctr += 1

    print(f'Epoch {epoch:02d}/{max_epochs} | tr_loss={tr_loss:.4f} vl_loss={vl_loss:.4f} | '
          f'val_f1={val_f1:.4f} | '
          f'branch acc: text={vl_t_acc:.4f} img={vl_i_acc:.4f} '
          f'(patience {patience_ctr}/{CONFIG["training"]["patience"]}){marker}')

    if patience_ctr >= CONFIG['training']['patience']:
        print(f'Early stopping at epoch {epoch}'); break

print(f'\nBest epoch: {best_epoch}  |  best val macro-F1: {best_val_f1:.4f}')

## Step 9: Test Evaluation


In [ ]:
eval_head = DecisionFusionHead(
    text_dim=CONFIG['model']['text_dim'],
    image_dim=CONFIG['model']['image_dim'],
    text_hidden=CONFIG['model']['text_hidden'],
    img_hidden=CONFIG['model']['img_hidden'],
    num_classes=CONFIG['model']['num_classes'],
    dropout=CONFIG['model']['dropout'],
).to(device)
_c = torch.load(best_ckpt_path, map_location=device)
eval_head.load_state_dict(_c['model_state_dict'])
eval_head.eval()
print(f'Loaded best checkpoint (epoch {_c["epoch"]})')

te_loss, te_main, te_aux, yt_te, yp_te, te_t_acc, te_i_acc = run_epoch(
    eval_head, loaders['test'], loss_fn, aux_loss_fn, None, None, device, False, aux_weight)

test_m = compute_metrics(yt_te, yp_te, 'test', nc)
test_m['test_loss']           = round(te_loss, 4)
test_m['test_text_branch_acc'] = round(te_t_acc, 4)
test_m['test_img_branch_acc']  = round(te_i_acc, 4)
test_m['confusion_matrix']    = confusion_matrix(yt_te, yp_te, labels=[0,1,2]).tolist()

print('\nTest results:')
for k, v in test_m.items():
    if k != 'confusion_matrix': print(f'  {k}: {v}')
print(f'  confusion_matrix: {test_m["confusion_matrix"]}')
print(f'\nBranch analysis (test):')
print(f'  Text branch alone accuracy : {te_t_acc:.4f}')
print(f'  Image branch alone accuracy: {te_i_acc:.4f}')
print(f'  Combined accuracy          : {test_m["test_accuracy"]:.4f}')
if test_m['test_accuracy'] > max(te_t_acc, te_i_acc):
    print('  \u2714 Fusion IMPROVES over best single branch')
else:
    print('  \u26a0 Fusion does NOT improve over best single branch')

# Save results JSON
results = {
    'notebook':  '06b_late_decision_fusion.ipynb',
    'proposal':  'D \u2014 Decision-Level Fusion',
    'run_name':  run_name,
    'best_epoch': best_epoch,
    'best_val_macro_f1': best_val_f1,
    'val_metrics':  {k: v for k, v in history[best_epoch-1].items() if k.startswith('val')},
    'test_metrics': test_m,
    'training_history': history,
    'arch': _cfg_jsonable(CONFIG['model']),
    'training': _cfg_jsonable(CONFIG['training']),
}
results_path = CONFIG['paths']['results_dir'] / f'{run_name}_results.json'
with open(results_path, 'w') as f: json.dump(results, f, indent=2)
print(f'\nResults saved: {results_path}')

if mlflow_enabled:
    try: mlflow.log_metrics({k: v for k, v in test_m.items() if k != 'confusion_matrix'}); mlflow.end_run()
    except Exception: pass

## Step 10: Training Curves + Branch Analysis


In [ ]:
if history:
    epochs = [r['epoch'] for r in history]
    color = '#10B981'

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (tr_k, vl_k, title) in zip(axes, [
        ('train_loss',     'val_loss',     'Loss'),
        ('train_accuracy', 'val_accuracy', 'Accuracy'),
        ('train_macro_f1', 'val_macro_f1', 'Macro F1'),
    ]):
        ax.plot(epochs, [r[tr_k] for r in history], label='train', color=color, alpha=0.7)
        ax.plot(epochs, [r[vl_k] for r in history], label='val',   color=color, linestyle='--')
        ax.axvline(best_epoch, color='gray', linestyle=':', label=f'best ep {best_epoch}')
        ax.set_title(f'06b Decision Fusion \u2014 {title}'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = CONFIG['paths']['results_dir'] / f'{run_name}_curves.png'
    plt.savefig(p, dpi=120); plt.show(); plt.close()

    # Branch accuracy over training (val)
    if 'val_text_branch_acc' in history[0]:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(epochs, [r['val_text_branch_acc'] for r in history],  label='text branch acc',  color='#7C3AED', linewidth=2)
        ax.plot(epochs, [r['val_img_branch_acc']  for r in history],  label='image branch acc', color='#F59E0B', linewidth=2)
        ax.plot(epochs, [r['val_accuracy']         for r in history],  label='combined acc',     color=color,    linewidth=2, linestyle='--')
        ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7)
        ax.set_title('Branch Accuracy vs Combined — val set')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = CONFIG['paths']['results_dir'] / f'{run_name}_branch_analysis.png'
        plt.savefig(p, dpi=120); plt.show(); plt.close()
        print(f'Branch analysis plot saved: {p}')

if test_m.get('confusion_matrix'):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(np.array(test_m['confusion_matrix']), annot=True, fmt='d', cmap='Greens',
                xticklabels=['Real', 'Fake', 'NEI'], yticklabels=['Real', 'Fake', 'NEI'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Test Confusion Matrix \u2014 06b Decision Fusion')
    plt.tight_layout()
    p = CONFIG['paths']['results_dir'] / f'{run_name}_confusion_matrix.png'
    plt.savefig(p, dpi=120); plt.show(); plt.close()
    print(f'Confusion matrix saved: {p}')

## Step 11: Comparison vs Baselines


In [ ]:
BASELINES = {
    '05a MIL Attention':       {'test_accuracy': None, 'test_macro_f1': None, 'test_f1_fake': None},
    '05b Asym Gate+Residual':  {'test_accuracy': None, 'test_macro_f1': None, 'test_f1_fake': None},
    '05d Misinfo-Aware MIL':   {'test_accuracy': None, 'test_macro_f1': None, 'test_f1_fake': None},
    '06a Cross-Attention':     {'test_accuracy': None, 'test_macro_f1': None, 'test_f1_fake': None},
}

for tag, results_dir_name in [
    ('05a MIL Attention',      'stage05a_results'),
    ('05b Asym Gate+Residual', 'stage05b_results'),
    ('05d Misinfo-Aware MIL',  'stage05d_results'),
    ('06a Cross-Attention',    'stage06a_results'),
]:
    d = DATA_ROOT / 'training' / results_dir_name
    if d.exists():
        cands = sorted(d.glob('*_results.json'), key=lambda p: p.stat().st_mtime, reverse=True)
        if cands:
            with open(cands[0]) as f: r = json.load(f)
            tm = r.get('test_metrics', {})
            BASELINES[tag] = {'test_accuracy': tm.get('test_accuracy'),
                               'test_macro_f1': tm.get('test_macro_f1'),
                               'test_f1_fake':  tm.get('test_f1_fake')}

print('='*62)
print(f'{"Model":<30} {"Acc":>8} {"F1":>8} {"F1-Fake":>10}')
print('-'*62)
for name, m in BASELINES.items():
    acc  = f'{m["test_accuracy"]:.4f}' if m['test_accuracy'] else '  N/A  '
    f1   = f'{m["test_macro_f1"]:.4f}' if m['test_macro_f1'] else '  N/A  '
    f1fk = f'{m["test_f1_fake"]:.4f}'  if m['test_f1_fake']  else '  N/A  '
    print(f'{name:<30} {acc:>8} {f1:>8} {f1fk:>10}')

print('-'*62)
print(f'{"06b Decision Fusion (ours)":<30} '
      f'{test_m["test_accuracy"]:>8.4f} '
      f'{test_m["test_macro_f1"]:>8.4f} '
      f'{test_m.get("test_f1_fake", 0):>10.4f}')
print('='*62)
print(f'\nText branch alone: acc={te_t_acc:.4f}  Image branch alone: acc={te_i_acc:.4f}')